# Catalog Lifecycle with Presets

**Persona:** Platform Administrator (sysadmin)

**Goal:** Demonstrate the end-to-end catalog lifecycle using presets and standard OGC protocols:

1. Create a catalog via the STAC API
2. Poll readiness via `GET /admin/catalogs/{id}` until `provisioning_status=ready`
3. Apply the `public_open_data` preset (`POST /configs/catalogs/{cat}/presets/public_open_data`)
4. Create a collection
5. Ingest 3 STAC items via STAC Transactions (`POST /stac/catalogs/{cat}/collections/{col}/items`)
6. Read back via STAC search (`POST /stac/catalogs/{cat}/search`)
7. Read back via OGC Features (`GET /features/catalogs/{cat}/collections/{col}/items`)
8. Teardown

**Preset used:** `public_open_data` — composite preset that bundles role seed, IAM policies,
public-catalog routing, STAC, web, and admin interfaces in the correct dependency order.
Defined at `modules/storage/presets/composites/public_open_data.py`, `tier=PLATFORM`,
`catalog_scopable=True`.

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_SYSADMIN_TOKEN`

**Optional:** `DYNASTORE_TOKEN` (bearer token for read operations)


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

SYSADMIN_TOKEN = os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")
CATALOG_ID = f"nb01-{uuid.uuid4().hex[:8]}"
COLLECTION_ID = "sentinel2-east-africa"

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=120.0)

print(f"BASE_URL    = {BASE_URL}")
print(f"CATALOG_ID  = {CATALOG_ID}")
print(f"COLLECTION  = {COLLECTION_ID}")
print(f"sysadmin    = {'set' if SYSADMIN_TOKEN else 'NOT SET'}")

if not SYSADMIN_TOKEN:
    raise RuntimeError(
        "DYNASTORE_SYSADMIN_TOKEN is not set (and IDP_* auto-mint did not produce one). "
        "Set it before running — every step below needs sysadmin rights."
    )

## Step 1 — Create catalog

POST a minimal STAC catalog payload. The platform creates an isolated PostgreSQL schema
on first write and returns `201 Created` once the record is persisted.


In [ ]:
r = client.post(
    "/stac/catalogs",
    content=json.dumps({
        "id": CATALOG_ID,
        "type": "Catalog",
        "title": "NB01 Lifecycle Test",
        "description": "Ephemeral catalog for the preset lifecycle walkthrough.",
        "stac_version": "1.0.0",
    }),
)
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"
print(f"Catalog created: {r.json()['id']}")


## Step 2 — Poll readiness

The platform may start async provisioning tasks after catalog creation. Poll
`GET /admin/catalogs/{id}` until `provisioning_status` is `ready` before applying a preset.
On-prem deployments with no external provisioners reach `ready` immediately.


In [ ]:
for _attempt in range(30):
    rr = client.get(f"/admin/catalogs/{CATALOG_ID}")
    if rr.status_code == 200 and rr.json().get("provisioning_status") == "ready":
        print(f"Catalog ready after {_attempt} polls")
        break
    time.sleep(2)
else:
    print("WARNING: catalog still provisioning after 60s — proceeding anyway")


## Step 3 — Discover and apply preset

First discover available presets, then apply `public_open_data` at the catalog scope.
The preset bundles: role seed, IAM policies, public-catalog routing, STAC, web, and admin
interfaces — all validated through the standard `ConfigsProtocol.set_config` lifecycle.

Route: `POST /configs/catalogs/{catalog_id}/presets/{preset_name}`


In [ ]:
# Discover available presets
r = client.get("/configs/presets")
print(f"GET /configs/presets -> {r.status_code}")
if r.status_code == 200:
    presets = r.json().get("presets", [])
    print(f"Available presets ({len(presets)}):")
    for p in presets[:10]:
        print(f"  - {p.get('name')}  tier={p.get('tier')}")


In [ ]:
# Apply public_open_data preset at catalog scope
r = client.post(
    f"/configs/catalogs/{CATALOG_ID}/presets/public_open_data",
    content="{}",
)
print(r.status_code, r.text[:400])
assert r.status_code in (200, 201, 204), (
    f"Preset apply expected 200/201/204, got {r.status_code}: {r.text}"
)
print("Preset 'public_open_data' applied successfully")


## Step 4 — Create collection

Create a STAC collection inside the catalog. The preset has already wired the
storage routing so the collection inherits the preset-configured drivers.


In [ ]:
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({
        "id": COLLECTION_ID,
        "type": "Collection",
        "stac_version": "1.0.0",
        "title": "Sentinel-2 East Africa",
        "description": "Test collection for the lifecycle notebook.",
        "license": "proprietary",
        "extent": {
            "spatial": {"bbox": [[28.0, -5.0, 42.0, 12.0]]},
            "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
        },
        "links": [],
    }),
)
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"
print(f"Collection '{COLLECTION_ID}' created")


## Step 5 — Ingest STAC items

POST 3 items via STAC Transactions. Each item is a minimal Sentinel-2 scene over
East Africa. A `201` response confirms the item was stored.

Route: `POST /stac/catalogs/{cat}/collections/{col}/items`


In [ ]:
ITEMS = [
    {
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": "S2A_20240115_T37MBN",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[33.0, 3.0], [36.0, 3.0], [36.0, 6.0], [33.0, 6.0], [33.0, 3.0]]],
        },
        "bbox": [33.0, 3.0, 36.0, 6.0],
        "properties": {
            "datetime": "2024-01-15T10:30:00Z",
            "platform": "sentinel-2a",
            "eo:cloud_cover": 12.5,
        },
        "assets": {},
        "links": [],
    },
    {
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": "S2B_20240122_T37MBN",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[33.0, 3.0], [36.0, 3.0], [36.0, 6.0], [33.0, 6.0], [33.0, 3.0]]],
        },
        "bbox": [33.0, 3.0, 36.0, 6.0],
        "properties": {
            "datetime": "2024-01-22T10:15:00Z",
            "platform": "sentinel-2b",
            "eo:cloud_cover": 4.1,
        },
        "assets": {},
        "links": [],
    },
    {
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": "S2A_20240129_T37MBN",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[33.0, 3.0], [36.0, 3.0], [36.0, 6.0], [33.0, 6.0], [33.0, 3.0]]],
        },
        "bbox": [33.0, 3.0, 36.0, 6.0],
        "properties": {
            "datetime": "2024-01-29T10:45:00Z",
            "platform": "sentinel-2a",
            "eo:cloud_cover": 28.3,
        },
        "assets": {},
        "links": [],
    },
]

ingested_ids = []
for item in ITEMS:
    r = client.post(
        f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
        content=json.dumps(item),
    )
    assert r.status_code == 201, (
        f"Expected 201 for {item['id']}, got {r.status_code}: {r.text[:300]}"
    )
    ingested_ids.append(r.json().get("id", item["id"]))
    print(f"Ingested: {ingested_ids[-1]}")

print(f"\nTotal ingested: {len(ingested_ids)}")


## Step 6 — STAC search readback

POST a STAC search to confirm items are queryable. Filter by collection and
verify all 3 ingested items are returned.

Route: `POST /stac/catalogs/{cat}/search`


In [ ]:
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/search",
    content=json.dumps({
        "collections": [COLLECTION_ID],
        "limit": 10,
    }),
)
print(f"STAC search -> {r.status_code}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text[:400]}"

result = r.json()
features = result.get("features", [])
returned = result.get("context", {}).get("returned", len(features))
print(f"Returned: {returned}")
for f in features:
    print(f"  - {f['id']}  cloud={f['properties'].get('eo:cloud_cover')}%")

assert len(features) >= len(ingested_ids), (
    f"Expected at least {len(ingested_ids)} items, got {len(features)}"
)
print("STAC search verified.")


## Step 7 — OGC Features readback

Retrieve items via the OGC API - Features endpoint. Confirms the preset wired
the Features protocol correctly alongside STAC.

Route: `GET /features/catalogs/{cat}/collections/{col}/items`


In [ ]:
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 10},
)
print(f"OGC Features -> {r.status_code}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text[:400]}"

fc = r.json()
features = fc.get("features", [])
print(f"numberMatched: {fc.get('numberMatched', 'n/a')}")
for f in features:
    props = f.get("properties", {})
    print(f"  - {f['id']}  platform={props.get('platform')}")

assert len(features) >= len(ingested_ids), (
    f"Expected at least {len(ingested_ids)} items, got {len(features)}"
)
print("OGC Features readback verified.")


## Teardown

Force-delete the test catalog. The `force=true` parameter cascades through
all collections, items, and the PostgreSQL schema for this catalog.


In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
print(r.status_code, r.text[:300])
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Catalog {CATALOG_ID} deleted.")
client.close()
